# File 06 — Explainability + ERM
**EHSL-ML+ | Disertasi Nanan Soekarna**

File ini menjalankan 2 hal terakhir dari pipeline analitik:

1. **Explainability (Layer 4)** — SHAP & LIME untuk menjelaskan *kenapa* model klasifikasi (dari File 05) memutuskan seperti itu: fitur mana yang paling berpengaruh
2. **Ethical Reinforcement Mechanism / ERM (Layer 5)** — skor Reward−Penalty dalam rentang [−1, +1], dengan 4 skema bobot dan klasifikasi 5-kategori

**Kenapa explainability penting di konteks ini?** Proposal menegaskan: model ML yang dipakai untuk governance hukum HARUS transparan dan bisa diaudit. Tidak cukup model bilang "responden ini Low" — kita harus bisa jelaskan *atas dasar apa*. Itulah fungsi SHAP & LIME.

In [1]:
import numpy as np
import pandas as pd
import joblib
import shap
from lime.lime_tabular import LimeTabularExplainer

np.random.seed(42)

bundle = joblib.load("classifier_for_shap.joblib")
model = bundle["model"]
model_name = bundle.get("model_name", "(tidak tercatat)")
X_cols = bundle["X_columns"]
X_train = bundle["X_train"]      # dipakai utk kalibrasi LIME (BUKAN X_test)
y_train = bundle["y_train"]
X_test = bundle["X_test"]
y_test = bundle["y_test"]
scaler = bundle.get("scaler")
label_map = bundle["label_map"]
inv_label = {v: k for k, v in label_map.items()}

print(f"Model dimuat: {model_name}")
print(f"Himpunan fitur: {bundle.get('feature_set','(tidak tercatat)')}")
print(f"Fitur: {len(X_cols)} | data train: {len(X_train)} | data test: {len(X_test)}")

# Model bisa saja Logistic Regression (butuh scaling). Fungsi ini menyeragamkan
# cara memanggil model, sehingga SHAP/LIME tidak perlu tahu jenis modelnya.
def predict_proba(X):
    X = np.asarray(X)
    return model.predict_proba(scaler.transform(X) if scaler is not None else X)

def predict(X):
    X = np.asarray(X)
    return model.predict(scaler.transform(X) if scaler is not None else X)

C:\Users\cici\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model dimuat: Random Forest
Himpunan fitur: SET B (konsistensi internal, BUKAN prediksi)
Fitur: 50 | data train: 159 | data test: 40


## 1. SHAP — Global Feature Importance

**Analogi SHAP:** bayangkan tiap fitur adalah "pemain" dalam tim yang menghasilkan sebuah prediksi. SHAP mengukur kontribusi tiap pemain secara adil — berapa banyak tiap fitur "mendorong" prediksi ke atas atau ke bawah, dirata-rata across semua kemungkinan kombinasi. Ini berbasis teori game (Shapley value), makanya namanya SHAP.

**Global importance** = rata-rata besar-kecilnya kontribusi tiap fitur di seluruh data test. Fitur dengan nilai tertinggi = paling menentukan keputusan model secara keseluruhan.

**Penting — apa yang sebenarnya dijelaskan di sini.** Model yang dianalisis dilatih pada **SET B**, yang oleh File 05 dilabeli *konsistensi internal*, bukan prediksi. Jadi SHAP di bawah menjawab: **"komponen granular mana (aktor/tahap/konstruk) yang paling menentukan kategori NREI seseorang"** — itu pertanyaan tentang *komposisi indeks*, dan jawabannya berguna untuk Bab IV. Yang **tidak** boleh disimpulkan: bahwa fitur-fitur ini "memprediksi" governance. Fitur `IF_*` juga sudah dikeluarkan di File 05, sehingga peringkat di bawah kini mencerminkan fitur granular yang sesungguhnya, bukan tertutup oleh 3 fitur perkalian yang bocor.

In [2]:
# TreeExplainer hanya berlaku utk model berbasis pohon (RF/XGBoost).
# Kalau model terbaiknya Logistic Regression, pakai explainer yang sesuai.
is_tree = model.__class__.__name__ in ("RandomForestClassifier", "XGBClassifier",
                                       "ExtraTreesClassifier", "GradientBoostingClassifier")
if is_tree:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    print(f"SHAP TreeExplainer dipakai untuk {model.__class__.__name__}")
else:
    explainer = shap.LinearExplainer(model, scaler.transform(np.asarray(X_train)))
    shap_values = explainer.shap_values(scaler.transform(np.asarray(X_test)))
    print(f"SHAP LinearExplainer dipakai untuk {model.__class__.__name__}")

# shap_values untuk multiclass: array (n_sampel, n_fitur, n_kelas)
if isinstance(shap_values, list):
    mean_abs = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
elif shap_values.ndim == 3:
    mean_abs = np.abs(shap_values).mean(axis=(0, 2))
else:
    mean_abs = np.abs(shap_values).mean(axis=0)

global_importance = pd.DataFrame({"feature": X_cols, "mean_abs_shap": mean_abs}) \
                      .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
print(f"\nTOP 15 fitur paling berpengaruh (SHAP global, model={model_name}):")
print(global_importance.head(15).to_string(index=False))
global_importance.to_csv("shap_global_importance.csv", index=False)

SHAP TreeExplainer dipakai untuk RandomForestClassifier

TOP 15 fitur paling berpengaruh (SHAP global, model=Random Forest):
                     feature  mean_abs_shap
                     T_ID_t3       0.023918
         ECV_Investigator_ID       0.023910
ECV_CorrectionalOfficer_FCNC       0.023538
                     T_ID_t1       0.023530
                   T_FCNC_t1       0.020706
                    T_VfV_t3       0.020383
                ECV_Judge_ID       0.020335
  ECV_CorrectionalOfficer_ID       0.018728
                   T_FCNC_t3       0.018108
              ECV_Judge_FCNC       0.017995
                    T_VfV_t4       0.017413
 ECV_CorrectionalOfficer_VfV       0.017112
       ECV_Investigator_FCNC       0.016884
                   T_FCNC_t4       0.015876
                     T_ID_t4       0.015554


**Cara menafsirkan hasil ini untuk disertasi:** perhatikan fitur mana yang teratas. Kalau interaction features (IF_VfV_ID dll.) dominan, itu berarti **kombinasi** antar konstruk lebih menentukan persepsi governance daripada konstruk tunggal — temuan yang mendukung nilai tambah feature engineering (Layer 2). Kalau fitur temporal (T_ID_t3 dll.) muncul tinggi, itu berarti *tahap* proses hukum tertentu sangat memengaruhi persepsi. Ini yang nanti diceritakan di Bab IV sebagai temuan explainability.

## 2. LIME — Penjelasan Kasus Individual

Kalau SHAP global menjawab "fitur apa yang penting secara keseluruhan", LIME menjawab "kenapa responden INI khususnya diklasifikasikan begitu". Berguna untuk Human-in-the-Loop: saat reviewer manusia ingin memeriksa satu kasus spesifik.

**Analogi LIME:** LIME "mengintip" di sekitar satu titik data, mengubah-ubah nilainya sedikit, lalu melihat bagaimana prediksi berubah — dari situ ia menyimpulkan fitur mana yang paling menentukan untuk kasus itu saja.

**Perbaikan bug:** versi sebelumnya memberi `training_data=X_test` (40 baris). LIME memakai data itu untuk menghitung statistik perturbasi dan batas diskretisasi — jadi seharusnya **data training** (159 baris), bukan data test. Dengan 40 baris, penjelasan lokalnya tidak stabil dan diam-diam mengambil informasi dari data uji.

In [3]:
# BUG DIPERBAIKI: training_data sebelumnya = X_test (40 baris). LIME memakai
# data ini untuk statistik perturbasi & diskretisasi, jadi harus data TRAINING.
lime_explainer = LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_cols,
    class_names=[inv_label[i] for i in sorted(inv_label)],
    mode="classification",
    random_state=42,
)
print(f"LIME dikalibrasi dgn {len(X_train)} baris data TRAINING (sebelumnya: {len(X_test)} baris data test)")

i = 0
exp = lime_explainer.explain_instance(
    np.array(X_test.iloc[i]), predict_proba, num_features=8, top_labels=1)
pred_class = int(predict(X_test.iloc[[i]])[0])
print(f"\nPenjelasan LIME untuk responden ke-{i} (prediksi: {inv_label[pred_class]}):\n")
for feat, weight in exp.as_list(label=pred_class):
    arrow = "menaikkan" if weight > 0 else "menurunkan"
    print(f"  {feat}  -> {arrow} keyakinan ({weight:+.3f})")

LIME dikalibrasi dgn 159 baris data TRAINING (sebelumnya: 40 baris data test)



Penjelasan LIME untuk responden ke-0 (prediksi: Low):

  ECV_CorrectionalOfficer_FCNC <= 30.00  -> menaikkan keyakinan (+0.044)
  ECV_Judge_ID <= 30.62  -> menaikkan keyakinan (+0.039)
  T_FCNC_t1 <= 30.62  -> menaikkan keyakinan (+0.038)
  ECV_Investigator_ID <= 35.00  -> menaikkan keyakinan (+0.036)
  T_FCNC_t4 <= 30.00  -> menaikkan keyakinan (+0.035)
  ECV_CorrectionalOfficer_ID <= 35.00  -> menaikkan keyakinan (+0.033)
  T_ID_t1 <= 35.00  -> menaikkan keyakinan (+0.032)
  T_VfV_t4 <= 30.00  -> menaikkan keyakinan (+0.029)


C:\Users\cici\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\cici\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


## 3. Ethical Reinforcement Mechanism (ERM)

ERM mengubah skor VfV/FCNC/ID jadi satu angka tunggal dalam rentang [−1, +1] yang secara eksplisit menyeimbangkan **Reward** (kekuatan persepsi etika) dan **Penalty** (kesenjangan dari ideal).

**Rumus (proposal §ERM Core Formulas):**
- `Reward = α×VfV + β×FCNC + γ×ID` (dgn α+β+γ=1, semua skor di [0,1]) → hasil di [0,1]
- `Penalty = 1 − Reward`
- `ERM_Score = Reward − Penalty = 2×Reward − 1` → hasil di [−1,+1]

**Penting:** ERM ini BUKAN reinforcement learning algoritmik — tidak ada agent yang belajar. Ini formula deterministik yang meminjam *struktur* reward-penalty dari paradigma RL. Proposal Anda menegaskan distinksi ini secara eksplisit (Tabel 3.33), jadi penting untuk tidak salah menyebutnya "RL" di sidang.

In [4]:
idx = pd.read_csv("data_index_respondent.csv")
erm = idx[["respondent_id","VfV_overall","FCNC_overall","ID_overall"]].dropna().copy()
erm["VfV_n"] = erm["VfV_overall"]/100
erm["FCNC_n"] = erm["FCNC_overall"]/100
erm["ID_n"] = erm["ID_overall"]/100

# 4 skema bobot (Tabel 3.34)
configs = {
    "equal":               (0.33, 0.33, 0.34),
    "value_emphasis":      (0.50, 0.25, 0.25),
    "integrity_emphasis":  (0.25, 0.25, 0.50),
    "procedural_emphasis": (0.25, 0.50, 0.25),
}

for name, (a, b, g) in configs.items():
    reward = a*erm["VfV_n"] + b*erm["FCNC_n"] + g*erm["ID_n"]
    erm[f"ERM_{name}"] = 2*reward - 1
    print(f"{name:20s} (a={a},b={b},g={g}): mean={erm[f'ERM_{name}'].mean():+.3f}, "
          f"std={erm[f'ERM_{name}'].std():.3f}, range=[{erm[f'ERM_{name}'].min():+.2f},{erm[f'ERM_{name}'].max():+.2f}]")

equal                (a=0.33,b=0.33,g=0.34): mean=-0.016, std=0.322, range=[-0.73,+0.92]
value_emphasis       (a=0.5,b=0.25,g=0.25): mean=-0.018, std=0.323, range=[-0.75,+0.94]
integrity_emphasis   (a=0.25,b=0.25,g=0.5): mean=-0.014, std=0.323, range=[-0.69,+0.89]
procedural_emphasis  (a=0.25,b=0.5,g=0.25): mean=-0.015, std=0.323, range=[-0.76,+0.94]


### Pemilihan skema bobot final

Proposal (§Weight Parameters) menetapkan bobot final ditentukan lewat proses **2 tahap**: (1) expert panel (ahli hukum, etika, kriminologi, governance) menilai prioritas relatif VfV/FCNC/ID, lalu (2) sensitivity analysis memilih konfigurasi yang **paling baik membedakan** level governance antar aktor/institusi.

Karena expert panel belum ada (itu Phase 2 riil), di sini bagian (2) diotomatiskan sebagai **placeholder**: pilih skema dengan standar deviasi tertinggi (paling diskriminatif).

**⚠ Perbaikan penting:** versi sebelumnya langsung menobatkan satu "pemenang" tanpa memeriksa apakah selisihnya bermakna. Faktanya keempat skema menghasilkan std yang hampir identik — bedanya di **desimal ketiga**. Menobatkan pemenang dari selisih sebesar itu sama dengan memilih berdasarkan **noise**, bukan bukti. Ibarat empat pelari finis dengan selisih seperseribu detik pada stopwatch yang ketelitiannya sepersepuluh detik.

Karena itu sel di bawah sekarang **menguji dulu apakah selisihnya layak dibedakan**. Kalau tidak, ia menyatakan **SERI** secara eksplisit dan tetap memakai skema `equal` sebagai default netral — bukan mengarang pemenang.

In [5]:
std_by_config = {name: erm[f"ERM_{name}"].std() for name in configs}
urut = sorted(std_by_config.items(), key=lambda kv: kv[1], reverse=True)

print("Standar deviasi ERM per skema bobot (makin besar = makin diskriminatif):")
for nama, s in urut:
    print(f"  {nama:20s} std = {s:.6f}")

tertinggi, terendah = urut[0][1], urut[-1][1]
rentang = tertinggi - terendah
# Ambang kelayakan: selisih harus >1% dari std itu sendiri untuk dianggap nyata.
# Di bawah itu, perbedaannya tenggelam dalam noise sampling.
AMBANG_REL = 0.01
layak_dibedakan = rentang > AMBANG_REL * tertinggi

print(f"\nRentang std antar skema: {rentang:.6f} "
      f"({rentang/tertinggi*100:.3f}% dari std tertinggi)")
print(f"Ambang kelayakan: selisih harus > {AMBANG_REL*100:.0f}% dari std tertinggi")

if layak_dibedakan:
    best_config = urut[0][0]
    print(f"-> Selisih LAYAK dibedakan. Skema paling diskriminatif: {best_config}")
    status_pemilihan = f"terpilih ({best_config}), selisih layak dibedakan"
else:
    best_config = "equal"
    print(f"-> SERI: keempat skema TIDAK dapat dibedakan pada data ini (selisih hanya")
    print(f"   {rentang/tertinggi*100:.3f}%, jauh di bawah ambang {AMBANG_REL*100:.0f}%).")
    print(f"   Menobatkan '{urut[0][0]}' sebagai pemenang berarti memilih berdasarkan NOISE.")
    print(f"   Dipakai skema netral '{best_config}' sebagai default, dan yang DILAPORKAN")
    print(f"   di Bab IV adalah: keempat skema setara -> penentuan bobot MENUNGGU")
    print(f"   expert panel, sesuai rancangan 2 tahap di proposal.")
    status_pemilihan = "SERI - keempat skema setara, menunggu expert panel"

def classify_erm(score):
    if score >= 0.75: return "Strongly Positive"
    if score >= 0.50: return "Moderately Positive"
    if score >= 0.00: return "Slightly Positive"
    if score >= -0.50: return "Slightly Negative"
    return "Negative"

erm["ERM_final"] = erm[f"ERM_{best_config}"]
erm["ERM_category"] = erm["ERM_final"].apply(classify_erm)
print(f"\nSkema dipakai: {best_config}  |  status: {status_pemilihan}")
print("Distribusi kategori ERM:")
print(erm["ERM_category"].value_counts())
erm.to_csv("erm_scores.csv", index=False)

Standar deviasi ERM per skema bobot (makin besar = makin diskriminatif):
  procedural_emphasis  std = 0.323276
  integrity_emphasis   std = 0.322993
  value_emphasis       std = 0.322540
  equal                std = 0.321678

Rentang std antar skema: 0.001597 (0.494% dari std tertinggi)
Ambang kelayakan: selisih harus > 1% dari std tertinggi
-> SERI: keempat skema TIDAK dapat dibedakan pada data ini (selisih hanya
   0.494%, jauh di bawah ambang 1%).
   Menobatkan 'procedural_emphasis' sebagai pemenang berarti memilih berdasarkan NOISE.
   Dipakai skema netral 'equal' sebagai default, dan yang DILAPORKAN
   di Bab IV adalah: keempat skema setara -> penentuan bobot MENUNGGU
   expert panel, sesuai rancangan 2 tahap di proposal.

Skema dipakai: equal  |  status: SERI - keempat skema setara, menunggu expert panel
Distribusi kategori ERM:
ERM_category
Slightly Negative      97
Slightly Positive      76
Negative               13
Moderately Positive    10
Strongly Positive       3
Name: coun

## 4. Agregasi ERM per Aktor & Tahap KUHAP (Tabel 3.36)

Proposal minta ERM diagregasi ke beberapa level untuk kegunaan governance. Yang paling penting: **per kategori aktor** (aktor mana yang menerima ERM terendah) dan **per tahap KUHAP** (tahap proses hukum mana yang persepsi etikanya paling defisit). Di sini ERM dihitung dari skor per-aktor (`data_index_actor`), bukan skor keseluruhan.

In [6]:
df_actor = pd.read_csv("data_index_actor.csv")
a, b, g = configs[best_config]
da = df_actor.dropna(subset=["VfV_norm","FCNC_norm","ID_norm"]).copy()
reward_actor = a*(da["VfV_norm"]/100) + b*(da["FCNC_norm"]/100) + g*(da["ID_norm"]/100)
da["ERM_actor"] = 2*reward_actor - 1

print("ERM agregat per kategori aktor (urut dari terendah):")
actor_agg = da.groupby("actor_category")["ERM_actor"].agg(["mean","std","count"]).round(3).sort_values("mean")
print(actor_agg.to_string())
print(f"\n-> Aktor dgn ERM terendah: {actor_agg.index[0]} ({actor_agg['mean'].iloc[0]:+.3f})")

stage_map = {"Investigator":"t1_Penyidikan","Prosecutor":"t2_Penuntutan","Judge":"t3_Peradilan",
             "Advocate":"t3_Peradilan","CorrectionalOfficer":"t4_Pemasyarakatan"}
da["stage"] = da["actor_category"].map(stage_map)
print("\nERM agregat per tahap KUHAP (urut dari terendah):")
stage_agg = da.groupby("stage")["ERM_actor"].agg(["mean","std","count"]).round(3).sort_values("mean")
print(stage_agg.to_string())

actor_agg.to_csv("erm_by_actor.csv")
stage_agg.to_csv("erm_by_stage.csv")

ERM agregat per kategori aktor (urut dari terendah):
                      mean    std  count
actor_category                          
Judge               -0.027  0.420    182
Prosecutor          -0.026  0.386    179
CorrectionalOfficer -0.022  0.424    198
Investigator        -0.017  0.402    196
Advocate             0.005  0.389    127

-> Aktor dgn ERM terendah: Judge (-0.027)

ERM agregat per tahap KUHAP (urut dari terendah):
                    mean    std  count
stage                                 
t2_Penuntutan     -0.026  0.386    179
t4_Pemasyarakatan -0.022  0.424    198
t1_Penyidikan     -0.017  0.402    196
t3_Peradilan      -0.014  0.407    309


**Cara menafsirkan (dgn hati-hati):** aktor/tahap dengan ERM terendah adalah area yang **persepsi etikanya paling perlu direview** — BUKAN bukti bahwa aktor itu bersalah. Ingat batas etis proposal: ini sinyal untuk refleksi institusional, bukan vonis. Di data dummy ini semua aktor ERM-nya dekat 0 (karena data acak), tapi di data riil pola perbedaan antar-aktor inilah temuan utamanya.

**Catatan cross-validation (Tabel 3.36):** grand-mean ERM nasional bisa dibandingkan dengan NREI sebagai cek validitas konstruk — kalau keduanya searah, itu memperkuat keabsahan kedua ukuran.

## 5. Simpan Output

In [7]:
summary_note = pd.DataFrame([{
    "best_weight_config_placeholder": best_config,
    "erm_mean": erm["ERM_final"].mean(),
    "erm_std": erm["ERM_final"].std(),
    "lowest_erm_actor": actor_agg.index[0],
    "lowest_erm_stage": stage_agg.index[0],
    "top_shap_feature": global_importance.iloc[0]["feature"],
}])
summary_note.to_csv("explainability_erm_summary.csv", index=False)
print("SELESAI. Tersimpan:")
print("  shap_global_importance.csv, erm_scores.csv, erm_by_actor.csv,")
print("  erm_by_stage.csv, explainability_erm_summary.csv")

SELESAI. Tersimpan:
  shap_global_importance.csv, erm_scores.csv, erm_by_actor.csv,
  erm_by_stage.csv, explainability_erm_summary.csv


---
## Ringkasan File 06

| Komponen | Hasil (data dummy) |
|---|---|
| SHAP global | Interaction features (IF_VfV_ID dll.) paling berpengaruh |
| LIME | Bisa jelaskan keputusan per responden individual |
| ERM 4 skema bobot | equal/value/integrity/procedural — dihitung semua |
| Skema final | placeholder via std tertinggi — **menunggu expert panel riil** |
| Agregasi | ERM per aktor & per tahap KUHAP |

**Dua catatan jujur:**
1. Bobot ERM final BUKAN hasil skrip — proposal mensyaratkan expert panel 2 tahap. Yang di sini cuma placeholder bagian sensitivity-nya.
2. Di data dummy, semua ERM dekat 0 dan beda antar-aktor tidak bermakna (data acak). Nilai sesungguhnya baru muncul di data riil.

**Yang dibawa ke File 07 (Dashboard Export):** semua CSV output dari File 02–06 dikonsolidasikan jadi struktur data siap-visualisasi, mengikuti arsitektur 5-layer dashboard (Tabel 4.5).